<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/dbmdz_mBERT_XLM_RoBERTa_twitter_XLM_(IT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, classification_report

In [3]:
df = pd.read_csv('/content/StereoQueerEval_IT_training.tsv', sep='\t')
df.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_IT_0001,"Nivan transgender a 17 anni: ""Il mio percorso ...",Nivan è un ragazzo transgender Ftm di soli 17 ...,Sono transgender ma ho paura di dirlo ai miei ...,no,no,none
1,training_IT_0002,"Casillas annuncia di essere gay, poi parla di ...","Travolto dalle polemiche, Casillas è stato cos...",E cosa sarà mai guardate cosa succede nel mond...,no,yes_implicit,group_g
2,training_IT_0003,Schlein: «Con governo Meloni l’Italia è scivol...,«Sono sempre felice quando c’è il riconoscimen...,Ma fate silenzio il gay pride e finito ora and...,yes,yes_explicit,group_lgbtqia+
3,training_IT_0004,"Olimpiadi Parigi 2024, la pugile Imane Khelif ...",Dopo che Angela Carini alle Olimpiadi di Parig...,Donna Trans NON ESISTE! Un uomo non puo` diven...,yes,yes_explicit,group_t
4,training_IT_0005,"Roma Pride 2018, l'inno ""Ma che ti importa"": ""...",Sabato 9 giugno il Roma Pride scende in piazza...,Allora che vi importa se uno dice che prenderl...,no,yes_explicit,"group_g,lgbtqia+"


In [4]:
df["combined_text"] = (
    "Comment: " + df["yt_comment"].fillna("").astype(str)
    + " Title: " + df["yt_title"].fillna("").astype(str)
    + " Description: " + df["yt_description"].fillna("").astype(str)
)

In [5]:
df.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target,combined_text
0,training_IT_0001,"Nivan transgender a 17 anni: ""Il mio percorso ...",Nivan è un ragazzo transgender Ftm di soli 17 ...,Sono transgender ma ho paura di dirlo ai miei ...,no,no,none,Comment: Sono transgender ma ho paura di dirlo...
1,training_IT_0002,"Casillas annuncia di essere gay, poi parla di ...","Travolto dalle polemiche, Casillas è stato cos...",E cosa sarà mai guardate cosa succede nel mond...,no,yes_implicit,group_g,Comment: E cosa sarà mai guardate cosa succede...
2,training_IT_0003,Schlein: «Con governo Meloni l’Italia è scivol...,«Sono sempre felice quando c’è il riconoscimen...,Ma fate silenzio il gay pride e finito ora and...,yes,yes_explicit,group_lgbtqia+,Comment: Ma fate silenzio il gay pride e finit...
3,training_IT_0004,"Olimpiadi Parigi 2024, la pugile Imane Khelif ...",Dopo che Angela Carini alle Olimpiadi di Parig...,Donna Trans NON ESISTE! Un uomo non puo` diven...,yes,yes_explicit,group_t,Comment: Donna Trans NON ESISTE! Un uomo non p...
4,training_IT_0005,"Roma Pride 2018, l'inno ""Ma che ti importa"": ""...",Sabato 9 giugno il Roma Pride scende in piazza...,Allora che vi importa se uno dice che prenderl...,no,yes_explicit,"group_g,lgbtqia+",Comment: Allora che vi importa se uno dice che...


In [6]:
df['combined_text']

,combined_text
0,Comment: Sono transgender ma ho paura di dirlo...
1,Comment: E cosa sarà mai guardate cosa succede...
2,Comment: Ma fate silenzio il gay pride e finit...
3,Comment: Donna Trans NON ESISTE! Un uomo non p...
4,Comment: Allora che vi importa se uno dice che...
...,...
2395,Comment: Era solo questione di tempo perché ve...
2396,"Comment: Più che la bandiera arcobaleno , si d..."
2397,Comment: 2021????!!......che mondo di 💩 \nMi ...
2398,Comment: mi sembra niente male sia fisicamente...


In [62]:
df["stereotype"].value_counts(normalize=True)

,proportion
stereotype,
no,0.73
yes,0.27


In [7]:
X = list(df['combined_text'])
y = df['stereotype'].map({'no':0, 'yes':1}).tolist()

In [8]:
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=45)

In [9]:
!pip install transformers

In [10]:
from transformers import AutoTokenizer

### dbmdz

In [11]:
from transformers import AutoModel, AutoTokenizer

model_name = "dbmdz/bert-base-italian-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

In [12]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=256)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=256)

In [13]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [15]:
from transformers import AutoModelForSequenceClassification, Trainer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    load_best_model_at_end=True
)

trainer.train()

Using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-italian-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,0.657838
20,0.603267
30,0.538591
40,0.595376
50,0.665508
60,0.592085
70,0.707767
80,0.576441
90,0.614703
100,0.564560


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=480, training_loss=0.5654959311087926, metrics={'train_runtime': 192.6814, 'train_samples_per_second': 19.929, 'train_steps_per_second': 2.491, 'total_flos': 505173226291200.0, 'train_loss': 0.5654959311087926, 'epoch': 2.0})

In [16]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.550183,0.543921,480


{'eval_loss': 0.5439213514328003}

In [17]:
trainer.predict(test_dataset)

PredictionOutput(predictions=array([[ 1.02794981e+00, -1.26359034e+00],
       [ 1.29009759e+00, -1.54506922e+00],
       [ 3.07289362e-01, -3.20498757e-02],
       [ 4.17748868e-01, -2.89918989e-01],
       [ 5.48595414e-02,  2.94779509e-01],
       [ 4.60543185e-01, -4.53890204e-01],
       [ 1.20991957e+00, -1.44132483e+00],
       [ 1.06666708e+00, -1.31306696e+00],
       [ 1.01575589e+00, -1.23956668e+00],
       [ 1.11455071e+00, -1.28617549e+00],
       [-2.93604820e-03,  1.96918994e-01],
       [ 1.23157549e+00, -1.50396895e+00],
       [ 7.77113199e-01, -9.01506722e-01],
       [ 9.61698219e-02,  2.71896750e-01],
       [ 4.97258723e-01, -4.34713930e-01],
       [ 1.48294196e-01,  1.18519843e-01],
       [ 1.14017105e+00, -1.46075797e+00],
       [ 1.11181557e+00, -1.43498218e+00],
       [ 8.25823009e-01, -9.75259185e-01],
       [ 9.67929006e-01, -1.27017987e+00],
       [ 8.70228112e-01, -1.13947594e+00],
       [ 1.55955359e-01,  2.14611962e-01],
       [ 1.12313092e+00, 

In [18]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

In [20]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.77      0.88      0.82       359
           1       0.38      0.22      0.28       121

    accuracy                           0.71       480
   macro avg       0.58      0.55      0.55       480
weighted avg       0.67      0.71      0.68       480



In [21]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[315  44]
 [ 94  27]]


### mBERT

In [24]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

In [25]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=256)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=256)

In [26]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
    # eval_strategy="epoch",
    # save_strategy="epoch",
    # load_best_model_at_end=True
)

In [40]:
from transformers import AutoModelForSequenceClassification, Trainer

model = AutoModelForSequenceClassification.from_pretrained('bert-base-multilingual-cased').to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,0.636212
20,0.613609
30,0.547498
40,0.603731
50,0.697498
60,0.603821
70,0.721871
80,0.598505
90,0.607326
100,0.578402


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=480, training_loss=0.5995132088661194, metrics={'train_runtime': 213.9334, 'train_samples_per_second': 17.95, 'train_steps_per_second': 2.244, 'total_flos': 505173226291200.0, 'train_loss': 0.5995132088661194, 'epoch': 2.0})

In [41]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.695169,0.563711,480


{'eval_loss': 0.5637111663818359}

In [42]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

In [43]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.75      1.00      0.86       359
           1       0.00      0.00      0.00       121

    accuracy                           0.75       480
   macro avg       0.37      0.50      0.43       480
weighted avg       0.56      0.75      0.64       480



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### XLM-RoBERTa

In [44]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [45]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=256)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=256)

In [46]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [47]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [49]:
from transformers import AutoModelForSequenceClassification, Trainer

model = AutoModelForSequenceClassification.from_pretrained('xlm-roberta-base').to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,0.667971
20,0.647475
30,0.638732
40,0.652640
50,0.643084
60,0.618334
70,0.617720
80,0.587702
90,0.655030
100,0.610863


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=480, training_loss=0.606318806608518, metrics={'train_runtime': 248.0137, 'train_samples_per_second': 15.483, 'train_steps_per_second': 1.935, 'total_flos': 505173226291200.0, 'train_loss': 0.606318806608518, 'epoch': 2.0})

In [50]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.693224,0.564662,480


{'eval_loss': 0.5646620392799377}

In [51]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

In [52]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.75      1.00      0.86       359
           1       0.00      0.00      0.00       121

    accuracy                           0.75       480
   macro avg       0.37      0.50      0.43       480
weighted avg       0.56      0.75      0.64       480



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### twitter-xlm-roberta

In [53]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [54]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=256)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=256)

In [55]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [56]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [57]:
from transformers import AutoModelForSequenceClassification, Trainer
import torch

model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Step,Training Loss
10,0.766534
20,0.664164
30,0.593745
40,0.641057
50,0.702556
60,0.632720
70,0.662179
80,0.650402
90,0.627479
100,0.578281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=480, training_loss=0.6028887997070949, metrics={'train_runtime': 332.3648, 'train_samples_per_second': 11.554, 'train_steps_per_second': 1.444, 'total_flos': 505177762037760.0, 'train_loss': 0.6028887997070949, 'epoch': 2.0})

In [58]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.612348,0.538761,480


{'eval_loss': 0.5387605428695679}

In [59]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

In [60]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.74      0.98      0.85       359
           1       0.00      0.00      0.00       121

    accuracy                           0.74       480
   macro avg       0.37      0.49      0.42       480
weighted avg       0.56      0.74      0.63       480

